# Bhutanese ASR — Transformer (Whisper) Fine-tuning for **16 GB VRAM**
## Encoder–decoder Transformer, LoRA, compatible with the Large-v3 notebook layout

**Target hardware:** single GPU with **~16 GB VRAM** (e.g. RTX 4060 Ti 16GB, RTX 4080 Laptop, L4, A4000).

**Base model:** [`openai/whisper-medium`](https://huggingface.co/openai/whisper-medium) (~769M params) — same Whisper transformer family as Large-v3, but fits comfortably with LoRA + gradient checkpointing on 16 GB.

**Why not Large-v3 here?** Large-v3 fine-tuning typically needs **~24–32 GB** even with LoRA and small batches. Medium is the practical default for 16 GB while staying in the same code path as your 32 GB notebook.

| Setting | 32 GB notebook (`whisper-large-v3`) | This notebook (16 GB) |
|--------|-------------------------------------|-------------------------|
| Model | `whisper-large-v3` | `whisper-medium` |
| Batch | 8 × grad_accum 2 | **2** × grad_accum **8** (effective 16) |
| Precision | bf16 | bf16 if supported, else **fp16** |

**Optional upgrade path:** If you gain more VRAM, change `model_name` to `openai/whisper-large-v3` and lower batch size / enable 8-bit loading per Hugging Face docs.

## 1. Install dependencies

In [ ]:
%%bash
pip install -q \
    transformers>=4.40.0 \
    datasets>=2.18.0 \
    accelerate>=0.28.0 \
    peft>=0.10.0 \
    bitsandbytes>=0.43.0 \
    evaluate>=0.4.0 \
    jiwer>=3.0.0 \
    librosa>=0.10.0 \
    soundfile>=0.12.0 \
    pandas \
    tqdm \
    tensorboard \
    PyMuPDF \
    pytesseract \
    Pillow \
    torch>=2.2.0 \
    torchaudio>=2.2.0

echo "✅ All packages installed"

## 2. Imports and environment

In [ ]:
import os
import re
import json
import warnings
import unicodedata
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import torch
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device : {device}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)
print(f"⚙️  Training dtype: {TORCH_DTYPE}")

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"💾  VRAM   : {total_vram:.1f} GB")
    if total_vram < 14:
        print("⚠️  Under ~14 GB: set per_device_train_batch_size=1 and/or use whisper-small.")
else:
    print("⚠️  CUDA not available — training will be very slow on CPU.")

print("✅ Environment ready")

## 3. Configuration (16 GB profile)

In [ ]:
CFG = {
    "model_name": "openai/whisper-medium",
    "language": "dzongkha",
    "task": "transcribe",

    "use_lora": True,
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,

    "sample_rate": 16_000,
    "max_audio_len_s": 30,

    "output_dir": "./whisper-dzongkha-bhutan-medium-16gb",
    # Tuned for ~16 GB: small micro-batch + higher accumulation
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-4,
    "warmup_steps": 500,
    "max_steps": 8_000,
    "eval_steps": 500,
    "save_steps": 500,
    "logging_steps": 50,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
    "gradient_checkpointing": True,
    "dataloader_num_workers": 2,

    "data_dir": "./data/bhutan",
    "train_split": 0.85,
    "val_split": 0.10,
    "test_split": 0.05,

    "docs_dir": "./data/bhutan_docs",
    "ocr_lang": "dzo+eng",
}

Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["data_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["docs_dir"]).mkdir(parents=True, exist_ok=True)

print("📋 Configuration loaded")
print(json.dumps({k: v for k, v in CFG.items() if k != "output_dir"}, indent=2))

## 4. Document text extraction (same as 32 GB notebook)

In [ ]:
import fitz
import pytesseract
from PIL import Image
import io


def extract_text_from_pdf(pdf_path: str, ocr_lang: str = "dzo+eng") -> str:
    doc = fitz.open(pdf_path)
    all_text = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text").strip()
        if len(text) < 20:
            pix = page.get_pixmap(dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(
                img, lang=ocr_lang, config="--oem 1 --psm 3"
            ).strip()
        if text:
            all_text.append(text)
    doc.close()
    return "\n".join(all_text)


def extract_text_from_image(img_path: str, ocr_lang: str = "dzo+eng") -> str:
    img = Image.open(img_path)
    return pytesseract.image_to_string(img, lang=ocr_lang, config="--oem 1 --psm 3").strip()


def batch_extract_documents(docs_dir: str, ocr_lang: str = "dzo+eng") -> List[Dict]:
    docs_path = Path(docs_dir)
    supported_ext = {".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"}
    records = []
    files = [f for f in docs_path.rglob("*") if f.suffix.lower() in supported_ext]
    print(f"📂 Found {len(files)} document(s) in {docs_dir}")
    for fp in tqdm(files, desc="Extracting documents"):
        try:
            if fp.suffix.lower() == ".pdf":
                text = extract_text_from_pdf(str(fp), ocr_lang)
            else:
                text = extract_text_from_image(str(fp), ocr_lang)
            if text:
                records.append({"filename": fp.name, "text": text})
        except Exception as e:
            print(f"  ⚠️  Skipping {fp.name}: {e}")
    print(f"✅ Extracted text from {len(records)} document(s)")
    return records


extracted_docs = batch_extract_documents(CFG["docs_dir"], CFG["ocr_lang"])
corpus_path = Path(CFG["data_dir"]) / "bhutan_corpus.json"
with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(extracted_docs, f, ensure_ascii=False, indent=2)
print(f"💾 Corpus saved → {corpus_path}")

## 5. Data cleaning

In [ ]:
def basic_clean(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


for rec in extracted_docs:
    rec["text"] = basic_clean(rec["text"])

print("✅ Basic cleaning applied to all documents")
if extracted_docs:
    sample = extracted_docs[0]
    print(f"\n📄 Sample ({sample['filename']}) — first 300 chars:")
    print(sample["text"][:300])

## 6. Load and prepare audio dataset

In [ ]:
def load_local_audio_dataset(data_dir: str, sample_rate: int = 16_000) -> DatasetDict:
    import pandas as pd

    meta_path = Path(data_dir) / "metadata.csv"
    if not meta_path.exists():
        print("⚠️  metadata.csv not found — creating a minimal demo dataset")
        return _create_demo_dataset(data_dir, sample_rate)

    df = pd.read_csv(meta_path)
    df["audio"] = df["file_name"].apply(lambda fn: str(Path(data_dir) / "audio" / fn))
    dataset = Dataset.from_pandas(df[["audio", "sentence"]])
    dataset = dataset.cast_column("audio", Audio(sampling_rate=sample_rate))
    split1 = dataset.train_test_split(test_size=0.15, seed=SEED)
    split2 = split1["test"].train_test_split(test_size=0.34, seed=SEED)
    return DatasetDict({
        "train": split1["train"],
        "validation": split2["train"],
        "test": split2["test"],
    })


def _create_demo_dataset(data_dir: str, sample_rate: int) -> DatasetDict:
    audio_dir = Path(data_dir) / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)
    samples = []
    for i in range(20):
        duration = np.random.randint(2, 8)
        audio = np.random.randn(duration * sample_rate).astype(np.float32) * 0.01
        path = audio_dir / f"demo_{i:04d}.wav"
        sf.write(str(path), audio, sample_rate)
        samples.append({"audio": str(path), "sentence": f"དཔེར་མཚོན། {i}"})
    dataset = Dataset.from_list(samples)
    dataset = dataset.cast_column("audio", Audio(sampling_rate=sample_rate))
    split = dataset.train_test_split(test_size=0.3, seed=SEED)
    return DatasetDict({"train": split["train"], "validation": split["test"], "test": split["test"]})


raw_datasets = load_local_audio_dataset(CFG["data_dir"], CFG["sample_rate"])
print(raw_datasets)
print(f"\n📊 Train samples     : {len(raw_datasets['train'])}")
print(f"📊 Validation samples: {len(raw_datasets['validation'])}")
print(f"📊 Test samples      : {len(raw_datasets['test'])}")

## 7. Processor and Whisper transformer model

In [ ]:
print(f"⏳ Loading processor and model: {CFG['model_name']} ...")

processor = WhisperProcessor.from_pretrained(
    CFG["model_name"],
    language=CFG["language"],
    task=CFG["task"],
)
feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

model = WhisperForConditionalGeneration.from_pretrained(
    CFG["model_name"],
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=CFG["language"], task=CFG["task"]
)
model.config.suppress_tokens = []

if CFG["gradient_checkpointing"]:
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

total_params = sum(p.numel() for p in model.parameters())
print(f"\n📦 Model loaded")
print(f"   Total parameters : {total_params/1e9:.2f}B")
if torch.cuda.is_available():
    print(f"   VRAM allocated   : {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 8. LoRA on the transformer (encoder + decoder projections)

In [ ]:
if CFG["use_lora"]:
    lora_config = LoraConfig(
        r=CFG["lora_r"],
        lora_alpha=CFG["lora_alpha"],
        lora_dropout=CFG["lora_dropout"],
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"🔧 LoRA applied")
    print(f"   Trainable params : {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")
    if torch.cuda.is_available():
        print(f"   VRAM allocated   : {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("ℹ️  Full fine-tuning (LoRA disabled) — not recommended on 16 GB for medium.")

## 9. Preprocessing

In [ ]:
MAX_LABEL_LEN = 448


def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="np",
    ).input_features[0]
    batch["labels"] = tokenizer(
        basic_clean(batch["sentence"]),
        max_length=MAX_LABEL_LEN,
        truncation=True,
    ).input_ids
    return batch


print("⚙️  Preprocessing datasets...")
processed_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=raw_datasets["train"].column_names,
    num_proc=min(4, os.cpu_count() or 1),
    desc="Preprocessing",
)
print("✅ Preprocessing complete")
print(processed_datasets)

## 10. Data collator

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
print("✅ Data collator ready")

## 11. WER metric

In [ ]:
wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 2)}


print("✅ WER metric ready")

## 12. Training arguments (memory-safe defaults)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["per_device_train_batch_size"],
    per_device_eval_batch_size=CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CFG["gradient_accumulation_steps"],
    max_steps=CFG["max_steps"],
    learning_rate=CFG["learning_rate"],
    warmup_steps=CFG["warmup_steps"],
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    lr_scheduler_type="cosine",
    fp16=CFG["fp16"],
    bf16=CFG["bf16"],
    gradient_checkpointing=CFG["gradient_checkpointing"],
    logging_steps=CFG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CFG["eval_steps"],
    save_steps=CFG["save_steps"],
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    predict_with_generate=True,
    generation_max_length=MAX_LABEL_LEN,
    dataloader_num_workers=CFG["dataloader_num_workers"],
    dataloader_pin_memory=torch.cuda.is_available(),
    seed=SEED,
)

eff_bs = CFG["per_device_train_batch_size"] * CFG["gradient_accumulation_steps"]
print("✅ Training arguments configured")
print(f"   Effective batch size: {eff_bs}")

## 13. Train

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("🚀 Starting training...")
print(f"   Model  : {CFG['model_name']}")
print(f"   Lang   : {CFG['language']}  ({CFG['task']})")
print(f"   Steps  : {CFG['max_steps']}")
print("─" * 60)

train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
print("\n✅ Training complete!")
print(train_result.metrics)

## 14. Test set evaluation

In [ ]:
print("📊 Evaluating on test set...")
metrics = trainer.evaluate(
    eval_dataset=processed_datasets["test"],
    metric_key_prefix="test",
)
trainer.log_metrics("test", metrics)
trainer.save_metrics("test", metrics)
print("\n📈 Test Results:")
for k, v in metrics.items():
    print(f"   {k:<30}: {v}")

## 15. Save final model

In [ ]:
save_path = Path(CFG["output_dir"]) / "final"

if CFG["use_lora"]:
    print("🔗 Merging LoRA adapters into base model...")
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(str(save_path))
else:
    trainer.save_model(str(save_path))

processor.save_pretrained(str(save_path))
print(f"💾 Model saved → {save_path}")

## 16. Inference pipeline

In [ ]:
from transformers import pipeline

infer_dtype = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)

asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=str(save_path),
    chunk_length_s=30,
    stride_length_s=5,
    device=0 if torch.cuda.is_available() else -1,
    torch_dtype=infer_dtype,
    generate_kwargs={
        "language": CFG["language"],
        "task": CFG["task"],
        "num_beams": 5,
    },
)


def transcribe_audio(audio_path: str) -> str:
    return asr_pipe(audio_path)["text"]


def extract_language_from_doc_audio(doc_text: str, audio_path: Optional[str] = None) -> Dict:
    output = {"doc_text": basic_clean(doc_text)}
    if audio_path and Path(audio_path).exists():
        output["transcript"] = transcribe_audio(audio_path)
    return output


demo_audio = list((Path(CFG["data_dir"]) / "audio").glob("*.wav"))
if demo_audio:
    sample_audio = str(demo_audio[0])
    print(f"🎤 Transcribing: {sample_audio}")
    print(f"📝 Transcript: {transcribe_audio(sample_audio)}")
else:
    print("ℹ️  No audio files found for demo — add .wav files to data/bhutan/audio/")

if extracted_docs:
    sample_doc = extracted_docs[0]
    print(f"\n📄 Document: {sample_doc['filename']}")
    print(f"   Extracted text (first 200 chars): {sample_doc['text'][:200]}")

## 17. VRAM report

In [ ]:
def vram_report():
    if not torch.cuda.is_available():
        print("No CUDA GPU detected.")
        return
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total = props.total_memory / 1e9
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        free = total - reserved
        print(f"GPU {i} — {props.name}")
        print(f"  Total    : {total:.2f} GB")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved : {reserved:.2f} GB")
        print(f"  Free     : {free:.2f} GB")


vram_report()

---
## Summary (16 GB profile)

| Item | Value |
|------|--------|
| Transformer | Whisper **Medium** (`openai/whisper-medium`) |
| Fine-tuning | LoRA r=32, α=64 |
| Micro-batch | 2 × gradient accumulation 8 → **effective batch 16** |
| Precision | bf16 when supported; else fp16 |
| VRAM | Designed to stay within **~16 GB** with checkpointing + LoRA |

**If you hit OOM:** set `per_device_train_batch_size` to `1`, or switch `model_name` to `openai/whisper-small`, or shorten `max_audio_len_s` / filter long utterances.